# 5. Генератор фотонов

Для энергии `E` число фотонов разыгрывается из распределения Пуассона со средним `E * Y`. Направления изотропны. Генератор выдаёт массивы направлений пачками по 100 фотонов; последняя пачка может быть короче.

In [1]:
import numpy as np

class Source:
    def __init__(self, Y=10000, seed=None):
        if Y < 0:
            raise ValueError("Y должно быть неотрицательным")
        self.Y = Y
        self.rng = np.random.default_rng(seed)

    def generate_photons(self, E, batch_size=100):
        if E < 0:
            raise ValueError("Энергия должна быть неотрицательной")
        if batch_size <= 0:
            raise ValueError("Размер пачки должен быть положительным")

        photon_count = self.rng.poisson(lam=E * self.Y)

        for batch_start in range(0, photon_count, batch_size):
            current_batch_size = min(batch_size, photon_count - batch_start)
            cos_theta = self.rng.uniform(-1.0, 1.0, current_batch_size)
            phi = self.rng.uniform(0.0, 2 * np.pi, current_batch_size)
            sin_theta = np.sqrt(1.0 - cos_theta**2)
            directions = np.column_stack((
                sin_theta * np.cos(phi),
                sin_theta * np.sin(phi),
                cos_theta
            ))
            yield directions

In [2]:
source = Source(seed=42)
energy = 1.0
photon_batches = list(source.generate_photons(energy))
batch_sizes = [len(batch) for batch in photon_batches]
photon_count = sum(batch_sizes)

print(f"Энергия события: {energy} МэВ")
print(f"Среднее число фотонов E*Y: {energy * source.Y:.0f}")
print(f"Разыгранное число фотонов: {photon_count}")
print(f"Размеры пачек: {batch_sizes}")

assert all(size == 100 for size in batch_sizes[:-1])
assert not batch_sizes or 0 < batch_sizes[-1] <= 100
if photon_batches:
    all_directions = np.concatenate(photon_batches, axis=0)
    assert np.allclose(np.linalg.norm(all_directions, axis=1), 1.0)

Энергия события: 1.0 МэВ
Среднее число фотонов E*Y: 10000
Разыгранное число фотонов: 10085
Размеры пачек: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 85]
